# Module 6: Big Data Storage & Databases

**BigData Analytics Course — AKTN Magister Program**

---

## Learning Objectives
1. Compare relational databases, NoSQL databases, and Data Lakes.
2. Use **SQLite** (SQL/relational) for structured data operations.
3. Use **MongoDB-style document operations** with Python.
4. Understand **HDFS** architecture and key design decisions.
5. Work with **Parquet** and **JSON** columnar/semi-structured formats.
6. Design an appropriate storage architecture for a given use case.

**Estimated time:** 120 minutes  
**Prerequisites:** Modules 1–2

In [ ]:
# Install additional packages (pymongo is simulated locally; no server required)
# !pip install pymongo --quiet   # uncomment if using a real MongoDB instance
# !pip install pyarrow fastparquet --quiet

import sqlite3
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pyarrow as pa
import pyarrow.parquet as pq

TMP = Path('/tmp/bigdata_module6')
TMP.mkdir(exist_ok=True)

print("✅ Libraries loaded.")

## 1. Storage Landscape

```
┌──────────────────────────────────────────────────────────────────┐
│                Big Data Storage Landscape                        │
│                                                                  │
│  Relational DB        NoSQL DB           Data Lake / Warehouse   │
│  ┌──────────┐         ┌──────────┐        ┌────────────────────┐ │
│  │PostgreSQL│         │MongoDB   │        │  HDFS / S3 / GCS   │ │
│  │MySQL     │         │Cassandra │        │  Delta Lake        │ │
│  │SQLite    │         │Redis     │        │  Apache Hudi       │ │
│  │Oracle    │         │HBase     │        │  BigQuery/Redshift │ │
│  └──────────┘         └──────────┘        └────────────────────┘ │
│  Structured           Flexible schema     Any format (schema-on-read)│
│  ACID                 BASE                Scalable, cheap storage    │
└──────────────────────────────────────────────────────────────────┘
```

| Property | RDBMS | NoSQL | Data Lake |
|----------|-------|-------|-----------|
| Schema | Fixed (schema-on-write) | Flexible | Schema-on-read |
| Transactions | ACID | BASE (eventual) | Varies (Delta = ACID) |
| Scalability | Vertical | Horizontal | Horizontal (massive) |
| Query language | SQL | Varies | SQL / custom |
| Best for | Transactional, consistent | High-write, distributed | Analytics, ML, archiving |

## 2. SQLite — Relational Database

In [ ]:
# ── 2.1 Create schema and load data ─────────────────────────────────────────
DB_PATH = str(TMP / 'ecommerce.db')
conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()

# Create tables
cur.executescript("""
    DROP TABLE IF EXISTS categories;
    DROP TABLE IF EXISTS customers;
    DROP TABLE IF EXISTS orders;

    CREATE TABLE categories (
        cat_id   INTEGER PRIMARY KEY,
        name     TEXT NOT NULL,
        tax_rate REAL DEFAULT 0.10
    );

    CREATE TABLE customers (
        cust_id  INTEGER PRIMARY KEY,
        name     TEXT NOT NULL,
        region   TEXT NOT NULL,
        age      INTEGER
    );

    CREATE TABLE orders (
        order_id     INTEGER PRIMARY KEY,
        cust_id      INTEGER REFERENCES customers(cust_id),
        cat_id       INTEGER REFERENCES categories(cat_id),
        quantity     INTEGER NOT NULL,
        unit_price   REAL NOT NULL,
        discount_pct INTEGER DEFAULT 0,
        order_date   TEXT
    );

    CREATE INDEX idx_orders_cat  ON orders(cat_id);
    CREATE INDEX idx_orders_cust ON orders(cust_id);
""")

# Insert seed data
categories = [
    (1, 'Electronics', 0.11),
    (2, 'Clothing',    0.10),
    (3, 'Books',       0.05),
    (4, 'Home',        0.10),
    (5, 'Sports',      0.12),
]
cur.executemany("INSERT INTO categories VALUES (?,?,?)", categories)

rng = np.random.default_rng(42)
regions = ['North', 'South', 'East', 'West']
n_cust = 200
customers = [
    (i, f'Customer_{i}', rng.choice(regions), int(rng.integers(18, 70)))
    for i in range(1, n_cust + 1)
]
cur.executemany("INSERT INTO customers VALUES (?,?,?,?)", customers)

n_ord = 2000
orders = [
    (
        i,
        int(rng.integers(1, n_cust + 1)),
        int(rng.integers(1, 6)),
        int(rng.integers(1, 20)),
        round(float(rng.uniform(5, 500)), 2),
        int(rng.choice([0, 5, 10, 15, 20])),
        f'2024-{rng.integers(1, 13):02d}-{rng.integers(1, 29):02d}'
    )
    for i in range(1, n_ord + 1)
]
cur.executemany("INSERT INTO orders VALUES (?,?,?,?,?,?,?)", orders)
conn.commit()
print(f"Database created at {DB_PATH}")
print(f"  Categories: {cur.execute('SELECT COUNT(*) FROM categories').fetchone()[0]}")
print(f"  Customers : {cur.execute('SELECT COUNT(*) FROM customers').fetchone()[0]}")
print(f"  Orders    : {cur.execute('SELECT COUNT(*) FROM orders').fetchone()[0]}")

In [ ]:
# ── 2.2 Analytical SQL queries ───────────────────────────────────────────────

# Query 1: Revenue by category (JOIN + aggregation)
q1 = """
    SELECT
        c.name                                        AS category,
        COUNT(o.order_id)                             AS num_orders,
        ROUND(SUM(o.quantity * o.unit_price * (1 - o.discount_pct / 100.0)), 2) AS revenue
    FROM orders o
    JOIN categories c ON o.cat_id = c.cat_id
    GROUP BY c.name
    ORDER BY revenue DESC;
"""
print("Revenue by Category:")
print(pd.read_sql(q1, conn).to_string(index=False))

# Query 2: Monthly revenue trend
q2 = """
    SELECT
        SUBSTR(order_date, 1, 7)                             AS month,
        COUNT(*)                                             AS orders,
        ROUND(SUM(quantity * unit_price * (1 - discount_pct / 100.0)), 2) AS revenue
    FROM orders
    GROUP BY month
    ORDER BY month;
"""
monthly = pd.read_sql(q2, conn)
print("\nMonthly Revenue Trend (first 6 rows):")
print(monthly.head(6).to_string(index=False))

In [ ]:
# ── 2.3 Advanced SQL — window function (running total) ───────────────────────
q3 = """
    SELECT
        month,
        revenue,
        ROUND(SUM(revenue) OVER (ORDER BY month ROWS UNBOUNDED PRECEDING), 2) AS running_total
    FROM (
        SELECT
            SUBSTR(order_date, 1, 7) AS month,
            ROUND(SUM(quantity * unit_price * (1 - discount_pct / 100.0)), 2) AS revenue
        FROM orders
        GROUP BY month
    )
    ORDER BY month;
"""
running = pd.read_sql(q3, conn)

fig, ax1 = plt.subplots(figsize=(11, 4))
ax2 = ax1.twinx()
ax1.bar(running['month'], running['revenue'], color='#2196F3', alpha=0.7, label='Monthly')
ax2.plot(running['month'], running['running_total'], 'ro-', lw=2, label='Running total')
ax1.set_xlabel('Month')
ax1.set_ylabel('Monthly Revenue ($)', color='#2196F3')
ax2.set_ylabel('Running Total ($)', color='red')
ax1.tick_params(axis='x', rotation=45)
ax1.set_title('Monthly Revenue with Running Total')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.show()
conn.close()

## 3. Document Store — MongoDB-style with Python

We simulate a MongoDB-style document store using Python dicts (JSON). In production you would use `pymongo` with a running MongoDB instance.

### Why Documents?
- Flexible schema — different documents in the same collection can have different fields.
- Nested structures — no joins needed for related data.
- Horizontal scaling — sharding across nodes.

In [ ]:
# ── 3.1 Simulate a document collection ──────────────────────────────────────
import random
random.seed(42)

categories_list = ['Electronics', 'Clothing', 'Books', 'Home', 'Sports']
regions_list    = ['North', 'South', 'East', 'West']

def make_order_doc(order_id):
    category = random.choice(categories_list)
    qty      = random.randint(1, 20)
    price    = round(random.uniform(5, 500), 2)
    disc     = random.choice([0, 5, 10, 15, 20])
    return {
        '_id'       : order_id,
        'category'  : category,
        'region'    : random.choice(regions_list),
        'quantity'  : qty,
        'unit_price': price,
        'discount'  : disc,
        'revenue'   : round(qty * price * (1 - disc / 100), 2),
        'customer'  : {
            'id'    : random.randint(1, 200),
            'age'   : random.randint(18, 70),
            'tier'  : random.choice(['Bronze', 'Silver', 'Gold', 'Platinum']),
        },
        'tags'       : random.sample(['sale', 'new', 'eco', 'premium', 'local'], k=random.randint(0, 3)),
    }

# In-memory collection (list of dicts)
collection = [make_order_doc(i) for i in range(1, 501)]

print(f"Collection size: {len(collection)} documents")
print("\nSample document:")
print(json.dumps(collection[0], indent=2))

In [ ]:
# ── 3.2 Document queries (Python as query engine) ───────────────────────────

def find(collection, query: dict):
    """Simulate MongoDB find() with a simple filter dict."""
    results = []
    for doc in collection:
        match = True
        for key, value in query.items():
            keys = key.split('.')                         # support nested keys
            val  = doc
            for k in keys:
                val = val.get(k, None) if isinstance(val, dict) else None
            if isinstance(value, dict):
                if '$gt'  in value and not (val is not None and val > value['$gt']):
                    match = False
                if '$gte' in value and not (val is not None and val >= value['$gte']):
                    match = False
                if '$lt'  in value and not (val is not None and val < value['$lt']):
                    match = False
            else:
                if val != value:
                    match = False
        if match:
            results.append(doc)
    return results

# Query 1: Electronics orders with revenue > 1000
q1_docs = find(collection, {'category': 'Electronics', 'revenue': {'$gt': 1000}})
print(f"Electronics orders with revenue > $1,000: {len(q1_docs)}")

# Query 2: Gold/Platinum customers
gold_orders = [d for d in collection if d['customer']['tier'] in ('Gold', 'Platinum')]
print(f"Gold/Platinum customer orders: {len(gold_orders)}")

# Query 3: Orders tagged 'eco'
eco_orders = [d for d in collection if 'eco' in d['tags']]
print(f"Orders tagged 'eco': {len(eco_orders)}")

# Aggregate — average revenue by category
from collections import defaultdict
rev_by_cat = defaultdict(list)
for doc in collection:
    rev_by_cat[doc['category']].append(doc['revenue'])

print("\nAverage Revenue by Category:")
for cat, revs in sorted(rev_by_cat.items()):
    print(f"  {cat:<15} avg=${sum(revs)/len(revs):.2f}  count={len(revs)}")

## 4. File Formats — Parquet vs JSON vs CSV

In [ ]:
# ── 4.1 Generate a medium-sized DataFrame ───────────────────────────────────
rng = np.random.default_rng(42)
n = 100_000

big_df = pd.DataFrame({
    'id'          : range(1, n + 1),
    'category'    : rng.choice(categories_list, n),
    'region'      : rng.choice(regions_list, n),
    'quantity'    : rng.integers(1, 20, n),
    'unit_price'  : rng.uniform(5.0, 500.0, n).round(2),
    'discount_pct': rng.choice([0, 5, 10, 15, 20], n),
    'revenue'     : rng.uniform(10, 9000, n).round(2),
})

csv_path     = str(TMP / 'data.csv')
json_path    = str(TMP / 'data.json')
parquet_path = str(TMP / 'data.parquet')

# Write in three formats
t0 = time.perf_counter()
big_df.to_csv(csv_path, index=False)
t_csv_write = time.perf_counter() - t0

t0 = time.perf_counter()
big_df.to_json(json_path, orient='records', lines=True)
t_json_write = time.perf_counter() - t0

t0 = time.perf_counter()
big_df.to_parquet(parquet_path, index=False)
t_parq_write = time.perf_counter() - t0

# File sizes
def mb(path): return os.path.getsize(path) / 1_048_576

print(f"{'Format':<10} {'Write(s)':<10} {'Size(MB)':<12}")
print("-" * 32)
print(f"{'CSV':<10} {t_csv_write:.3f}s     {mb(csv_path):.2f} MB")
print(f"{'JSON':<10} {t_json_write:.3f}s     {mb(json_path):.2f} MB")
print(f"{'Parquet':<10} {t_parq_write:.3f}s     {mb(parquet_path):.2f} MB")

In [ ]:
# ── 4.2 Read performance comparison ─────────────────────────────────────────
RUNS = 3
times = {'CSV': [], 'JSON': [], 'Parquet': []}

for _ in range(RUNS):
    t0 = time.perf_counter()
    pd.read_csv(csv_path)
    times['CSV'].append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    pd.read_json(json_path, lines=True)
    times['JSON'].append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    pd.read_parquet(parquet_path)
    times['Parquet'].append(time.perf_counter() - t0)

print("Read performance (avg of 3 runs):")
for fmt, ts in times.items():
    print(f"  {fmt:<10} {sum(ts)/len(ts):.4f}s")

# ── 4.3 Column pruning — Parquet advantage ───────────────────────────────────
t0 = time.perf_counter()
pd.read_parquet(parquet_path, columns=['id', 'revenue'])  # only 2 columns
t_prune = time.perf_counter() - t0
print(f"\nParquet column-pruned read (2/7 cols): {t_prune:.4f}s")

## 5. HDFS Architecture (Conceptual)

```
┌─────────────────────────────────────────────────────────────────┐
│                   HDFS Architecture                             │
│                                                                 │
│  Client                                                         │
│    │                                                            │
│    │ 1. Request file location                                   │
│    ▼                                                            │
│  NameNode  ──────────────────────────────────────────────────── │
│  (metadata: file→block map)                                     │
│    │                                                            │
│    │ 2. Returns block locations                                 │
│    ▼                                                            │
│  DataNode1  DataNode2  DataNode3  DataNode4  ...                │
│  [Block A]  [Block A]  [Block B]  [Block B]                     │
│  [Block B]  [Block C]  [Block C]  [Block A]  ← 3× replication  │
│                                                                 │
│  3. Client reads/writes directly from/to DataNodes             │
└─────────────────────────────────────────────────────────────────┘
```

| Feature | Value |
|---------|-------|
| Default block size | 128 MB (configurable) |
| Replication factor | 3 (fault-tolerant) |
| NameNode HA | Active/Standby pair since Hadoop 2 |
| Optimised for | Large sequential reads (not random access) |
| Not suitable for | Many small files, low-latency random read |

## 6. Storage Architecture Decision Framework

Use this decision tree when choosing a storage technology:

In [ ]:
# Visualise the storage decision framework
fig, ax = plt.subplots(figsize=(12, 8))
ax.axis('off')

# Table data
rows = [
    ['Use Case', 'Data Type', 'Scale', 'Recommendation'],
    ['OLTP (transactions)', 'Structured', 'GB–TB', 'PostgreSQL / MySQL'],
    ['Analytics (OLAP)', 'Structured', 'TB–PB', 'Redshift / BigQuery / Hive'],
    ['User profiles / catalogues', 'Semi-structured', 'GB–TB', 'MongoDB / DynamoDB'],
    ['High-write IoT', 'Time-series', 'TB–PB', 'Cassandra / InfluxDB'],
    ['Session / cache', 'Key-Value', 'GB', 'Redis'],
    ['ML feature store', 'Mixed', 'TB–PB', 'Delta Lake / Apache Hudi'],
    ['Raw data archive', 'Any', 'PB+', 'HDFS / S3 / GCS'],
    ['Graph data', 'Graph', 'GB–TB', 'Neo4j / Amazon Neptune'],
    ['Full-text search', 'Text', 'GB–TB', 'Elasticsearch'],
]

col_widths = [0.22, 0.18, 0.12, 0.30]
row_height = 0.09
x_starts = [0.02, 0.24, 0.42, 0.54]

colors_header = '#1565C0'
colors_row    = ['#E3F2FD', '#FFFFFF']

for r_idx, row in enumerate(rows):
    y = 0.95 - r_idx * row_height
    bg = colors_header if r_idx == 0 else colors_row[r_idx % 2]
    fc = 'white' if r_idx == 0 else 'black'
    rect = plt.Rectangle((0, y - row_height + 0.01), 1.0, row_height - 0.01,
                           color=bg, transform=ax.transAxes, clip_on=False)
    ax.add_patch(rect)
    for c_idx, (cell, x) in enumerate(zip(row, x_starts)):
        fw = 'bold' if r_idx == 0 else 'normal'
        ax.text(x, y - 0.02, cell, transform=ax.transAxes,
                fontsize=9, color=fc, fontweight=fw, va='top')

ax.set_title('Storage Technology Selection Guide', fontsize=14, pad=10)
plt.tight_layout()
plt.show()

## 7. Summary

| Technology | Type | Best For |
|-----------|------|----------|
| SQLite / PostgreSQL | Relational | ACID transactions, structured data |
| MongoDB | Document | Flexible schema, nested data |
| Cassandra | Wide-column | High-write, time-series, IoT |
| Redis | Key-Value | Caching, sessions |
| HDFS / S3 | Distributed FS | Massive-scale raw data storage |
| Parquet / ORC | File format | Columnar analytics, data lake |
| Delta Lake | Lakehouse | ACID on data lake, ML pipelines |

## 📝 Exercises

1. **SQL:** Write a query on the SQLite database that finds the top-5 customers by total revenue, including their region and tier (tip: you'll need to join with a tier column you add).
2. **Document store:** Extend the document collection to include a `shipping_address` nested object and implement a function to find all orders shipped to a specific city.
3. **Formats:** Load a 500 MB CSV file from Google Drive, convert it to Parquet with Snappy compression, and compare query performance for a filtered aggregation.

---
⬅️ [Module 5 — Apache Spark & PySpark](Module_05_Apache_Spark_PySpark.ipynb)  
➡️ [Module 7 — Capstone Project](Module_07_Capstone_Project.ipynb)